---
title: Equilibrium Climate Sensitivity from CMIP6
author: Harsha R. Hampapura
tags:
  - origin:aws
  - platform:casper
  - dataset:cmip6
  - task:ecs
  - level:advanced
---
# Access CMIP6 zarr data from AWS using the osdf protocol and compute Equilibrium Climate Sensitivity (ECS)

[<span class="tag tag-origin">origin:aws</span>](/tag-index#tag-origin-aws) [<span class="tag tag-platform">platform:casper</span>](/tag-index#tag-platform-casper) [<span class="tag tag-dataset">dataset:cmip6</span>](/tag-index#tag-dataset-cmip6) [<span class="tag tag-task">task:ecs</span>](/tag-index#tag-task-ecs) [<span class="tag tag-level">level:advanced</span>](/tag-index#tag-level-advanced)

## Section 1: Introduction
- Load python packkages
- Load catalog url

In [1]:
from matplotlib import pyplot as plt
import xarray as xr
import numpy as np
import dask
from dask.diagnostics import progress
from tqdm.autonotebook import tqdm
import intake
import fsspec
import seaborn as sns
import re
import aiohttp
from dask_jobqueue import PBSCluster
import pandas as pd
from xhistogram.xarray import histogram

/tmp/ipykernel_3561/3451259092.py:6: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
# import fsspec.implementations.http as fshttp
from pelicanfs.core import OSDFFileSystem,PelicanMap 

In [3]:
import os
# Local scratch on the EC2 instance (replaces the Casper /lustre path)
scratch_dir = os.path.expanduser("~/scratch")
os.makedirs(scratch_dir, exist_ok=True)
#
osdf_catalog       = 'https://data.gdex.ucar.edu/d850001/catalogs/cmip6-osdf-zarr.json'
pangeo_aws_catalog = 'https://cmip6-pds.s3.amazonaws.com/pangeo-cmip6.json'

## Section 2: Select Dask Cluster

In [4]:
#Create a LocalCluster using this EC2 instance's own CPU/RAM
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(local_directory=scratch_dir)   # auto-sizes to the instance
client = Client(cluster)
client

/home/ubuntu/osdf-env/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44699 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:44699/status,
Dashboard: http://127.0.0.1:44699/status,Workers: 4
Total threads: 4,Total memory: 15.25 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:33275,Workers: 0
Dashboard: http://127.0.0.1:44699/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:34139,Total threads: 1
Dashboard: http://127.0.0.1:35725/status,Memory: 3.81 GiB
Nanny: tcp://127.0.0.1:45765,


In [5]:
# Scale the cluster and display cluster dashboard URL
n_workers =6
cluster.scale(n_workers)
client.wait_for_workers(n_workers = n_workers)
cluster

LocalCluster(3330b3f5, 'tcp://127.0.0.1:33275', workers=6, threads=6, memory=22.87 GiB)

## Section 3: Data Loading
- Load catalog and select data subset

In [6]:
col = intake.open_esm_datastore(osdf_catalog)
col

,unique
activity_id,18
institution_id,36
source_id,88
experiment_id,170
member_id,657
table_id,37
variable_id,709
grid_label,10
zstore,522217
dcpp_init_year,61


In [7]:
[eid for eid in col.df['experiment_id'].unique() if 'ssp' in eid]

['esm-ssp585-ssp126Lu',
 'ssp126-ssp370Lu',
 'ssp370-ssp126Lu',
 'ssp585',
 'ssp245',
 'ssp370-lowNTCF',
 'ssp370SST-ssp126Lu',
 'ssp370SST',
 'ssp370pdSST',
 'ssp370SST-lowCH4',
 'ssp370SST-lowNTCF',
 'ssp126',
 'ssp119',
 'ssp370',
 'esm-ssp585',
 'ssp245-nat',
 'ssp245-GHG',
 'ssp460',
 'ssp434',
 'ssp534-over',
 'ssp245-aer',
 'ssp245-stratO3',
 'ssp245-cov-fossil',
 'ssp245-cov-modgreen',
 'ssp245-cov-strgreen',
 'ssp245-covid',
 'ssp585-bgc']

In [8]:
query = dict(
    experiment_id=['abrupt-4xCO2','piControl'], # pick the `abrupt-4xCO2` and `piControl` forcing experiments
    table_id='Amon',                            # choose to look at atmospheric variables (A) saved at monthly resolution (mon)
    variable_id=['tas', 'rsut','rsdt','rlut'],  # choose to look at near-surface air temperature (tas) as our variable
    member_id = 'r1i1p1f1',                     # arbitrarily pick one realization for each model (i.e. just one set of initial conditions)
)

col_subset = col.search(require_all_on=["source_id"], **query)
col_subset.df.groupby("source_id")[
    ["experiment_id", "variable_id", "table_id"]
].nunique()

,experiment_id,variable_id,table_id
source_id,,,
ACCESS-CM2,2,4,1
ACCESS-ESM1-5,2,4,1
AWI-CM-1-1-MR,2,4,1
BCC-CSM2-MR,2,4,1
BCC-ESM1,2,4,1
CAMS-CSM1-0,2,4,1
CAS-ESM2-0,2,4,1
CESM2,2,4,1
CESM2-FV2,2,4,1


In [9]:
def drop_all_bounds(ds):
    """Drop coordinates like 'time_bounds' from datasets,
    which can lead to issues when merging."""
    drop_vars = [vname for vname in ds.coords
                 if (('_bounds') in vname ) or ('_bnds') in vname]
    return ds.drop_vars(drop_vars)

def open_dsets(df):
    """Open datasets from cloud storage and return an xarray dataset."""
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    dsets = [xr.open_zarr(fsspec.get_mapper(ds_url), consolidated=True,
                          zarr_format=2, decode_times=time_coder)
             .pipe(drop_all_bounds)
             for ds_url in df.zstore]
    try:
        return xr.merge(dsets, join='exact')
    except ValueError:
        return None

def open_delayed(df):
    """A dask.delayed wrapper around `open_dsets`.
    Allows us to open many datasets in parallel."""
    return dask.delayed(open_dsets)(df)

In [10]:
from collections import defaultdict

dsets = defaultdict(dict)
for group, df in col_subset.df.groupby(by=['source_id', 'experiment_id']):
    dsets[group[0]][group[1]] = open_delayed(df)

In [11]:
%time open_dsets(df)

CPU times: user 571 ms, sys: 64.4 ms, total: 636 ms
Wall time: 8.67 s


<xarray.Dataset> Size: 5GB
Dimensions:  (time: 6000, lat: 192, lon: 288)
Coordinates:
  * time     (time) object 48kB 0201-01-16 12:00:00 ... 0700-12-16 12:00:00
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
    height   float64 8B ...
Data variables:
    rsut     (time, lat, lon) float32 1GB dask.array<chunksize=(328, 192, 288), meta=np.ndarray>
    rsdt     (time, lat, lon) float32 1GB dask.array<chunksize=(498, 192, 288), meta=np.ndarray>
    tas      (time, lat, lon) float32 1GB dask.array<chunksize=(407, 192, 288), meta=np.ndarray>
    rlut     (time, lat, lon) float32 1GB dask.array<chunksize=(355, 192, 288), meta=np.ndarray>
Attributes: (12/51)
    Conventions:               CF-1.7 CMIP-6.2
    activity_id:               CMIP
    branch_method:             branch-restart from year 0201-01-01 of piContr...
    branch_time:               0.0
    branch_time_in_child:      430335.0
    branch_time_in_parent:     430335.0
    ...                        ...
    table_id:                  Amon
    table_info:                Creation Date:(24 July 2019) MD5:0bb394a356ef9...
    title:                     TaiESM1 output prepared for CMIP6
    tracking_id:               hdl:21.14100/cd4ff0f0-e4b1-4158-b18b-1d8f071cb...
    variable_id:               rsut
    variant_label:             r1i1p1f1

In [12]:
dsets_ = dask.compute(dict(dsets))[0]

## Section 4: Data Analysis
- Reduce data via Global Mean
- Grab some observations ?

In [13]:
def get_lat_name(ds):
    """Figure out what is the latitude coordinate for each dataset."""
    for lat_name in ['lat', 'latitude']:
        if lat_name in ds.coords:
            return lat_name
    raise RuntimeError("Couldn't find a latitude coordinate")

def global_mean(ds):
    """Return global mean of a whole dataset."""
    lat = ds[get_lat_name(ds)]
    weight = np.cos(np.deg2rad(lat))
    weight /= weight.mean()
    other_dims = set(ds.dims) - {'time'}
    return (ds * weight).mean(other_dims)

In [14]:
expts = ['piControl', 'abrupt-4xCO2']
expt_da = xr.DataArray(expts, dims='experiment_id',
                       coords={'experiment_id': expts})

dsets_aligned = {}

for k, v in tqdm(dsets_.items()):
    expt_dsets = v.values()
    if any([d is None for d in expt_dsets]):
        print(f"Missing experiment for {k}")
        continue

    for ds in expt_dsets:
        ds.coords['year'] = ds.time.dt.year - ds.time.dt.year[0]

    # workaround for
    # https://github.com/pydata/xarray/issues/2237#issuecomment-620961663
    dsets_ann_mean = [v[expt].pipe(global_mean).swap_dims({'time': 'year'}).drop_vars('time').coarsen(year=12).mean()
                      for expt in expts]

    # align everything with the 4xCO2 experiment
    dsets_aligned[k] = xr.concat(dsets_ann_mean, join='right',dim=expt_da)

  7%|██████▌                                                                                  | 3/41 [00:00<00:09,  4.18it/s]

Missing experiment for ACCESS-ESM1-5


 22%|███████████████████▌                                                                     | 9/41 [00:01<00:03, 10.24it/s]

Missing experiment for CAS-ESM2-0


 41%|████████████████████████████████████▍                                                   | 17/41 [00:02<00:02,  9.66it/s]

...

 54%|███████████████████████████████████████████████▏                                        | 22/41 [00:02<00:01, 11.66it/s]

Missing experiment for FIO-ESM-2-0
Missing experiment for GFDL-CM4


 83%|████████████████████████████████████████████████████████████████████████▉               | 34/41 [00:04<00:00,  9.75it/s]

Missing experiment for MPI-ESM-1-2-HAM


100%|████████████████████████████████████████████████████████████████████████████████████████| 41/41 [00:05<00:00,  8.10it/s]


In [ ]:
%%time
dsets_aligned_ = dask.compute(dsets_aligned)[0]

In [ ]:
source_ids = list(dsets_aligned_.keys())
source_da = xr.DataArray(source_ids, dims='source_id',coords={'source_id': source_ids})

big_ds = xr.concat([ds.reset_coords(drop=True) for ds in dsets_aligned_.values()],
                   dim=source_da)
big_ds

### Calculated Derived Variables

In [ ]:
big_ds['imbalance'] = big_ds['rsdt'] - big_ds['rsut'] - big_ds['rlut']

ds_mean = big_ds[['tas', 'imbalance']].sel(experiment_id='piControl').mean(dim='year')
ds_anom = big_ds[['tas', 'imbalance']] - ds_mean

# add some metadata
ds_anom.tas.attrs['long_name'] = 'Global Mean Surface Temp Anom'
ds_anom.tas.attrs['units'] = 'K'
ds_anom.imbalance.attrs['long_name'] = 'Global Mean Radiative Imbalance'
ds_anom.imbalance.attrs['units'] = 'W m$^{-2}$'

ds_anom

In [ ]:
# limit to the gregory 150-year period
first_150_years = slice(0, 149)
ds_anom.tas.sel(year=first_150_years).plot.line(col='source_id', x='year', col_wrap=5)

### Calculate ECS

In [ ]:
ds_abrupt = ds_anom.sel(year=first_150_years, experiment_id='abrupt-4xCO2').reset_coords(drop=True)

In [ ]:
def calc_ecs(ds):
    tas_1d = ds.tas.squeeze()
    imbalance_1d = ds.imbalance.squeeze()

    # Align and drop NaNs
    tas_1d, imbalance_1d = xr.align(tas_1d.dropna('year'), imbalance_1d.dropna('year'), join='inner')

    a, b = np.polyfit(tas_1d.values, imbalance_1d.values, 1)
    ecs = -0.5 * (b / a)
    return xr.DataArray(ecs, attrs={'units': 'K'})

In [ ]:
ds_abrupt['ecs'] = ds_abrupt.groupby('source_id').apply(calc_ecs)
ds_abrupt

In [ ]:
%%time
# Convert to DataFrame
df = ds_abrupt.to_dataframe().reset_index()

# Drop NaNs
df = df.dropna(subset=['tas', 'imbalance'])

# Set up FacetGrid
g = sns.FacetGrid(df, col="source_id", col_wrap=5, height=3.5)
g.map_dataframe(sns.scatterplot, x="tas", y="imbalance")

# Add Gregory fit line and ECS text
def plot_ecs(data, color, **kwargs):
    x = data['tas'].values
    y = data['imbalance'].values
    mask = ~np.isnan(x) & ~np.isnan(y)
    x, y = x[mask], y[mask]
    
    if len(x) < 2:
        return  # skip underpopulated panels
    
    a, b = np.polyfit(x, y, 1)
    ecs = -0.5 * b / a
    
    x_line = np.array([0, x.max()])
    y_line = np.polyval([a, b], x_line)
    
    plt.plot(x_line, y_line, 'k')
    plt.text(0.6 * x.max(), 0.6 * y.max(), f'ECS={ecs:2.2f}K', fontsize=10, weight='bold')
    plt.grid(True)

g.map_dataframe(plot_ecs)

# Optional: adjust layout
g.set_titles("{col_name}")
g.set_axis_labels("Global Mean Tas (K)", "TOA Imbalance (W/m²)")
plt.tight_layout()
plt.show()


In [ ]:
cluster.close()